<a href="https://colab.research.google.com/github/a-goulart/mrr-archive-navigation/blob/main/finetuning_distilbert/DistilBERT_Fine_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook contains the steps taken to fine-tune the DistilBERT model.

# Loading dataset

The first step is to load the opinion columns

In [ ]:
import pandas as pd

In [ ]:
df_columns = pd.read_excel("/content/finalized_joined_columns.xlsx")

In [ ]:
df_columns.info()

In [ ]:
df_columns.head(5)

In total there are 274 unique opinion columns, from three different years: 1984, 1989 and 1994.

How many words am I working with?

In [ ]:
import re

df_columns['Text'] = df_columns['Text'].str.replace('\n', ' ', regex=False)

df_columns['Text'] = df_columns['Text'].str.replace(r'\s+', ' ', regex=True).str.strip()

In [ ]:
df_columns['totalwords'] = df_columns['Text'].str.split().str.len()

print(df_columns['totalwords'].sum())

In [ ]:
df_columns.head(5)

Ok, so a total of 407533 words

Since Transformer models have a max input length of 512 tokens, I will look at the word count distribution.

In [ ]:
import matplotlib.pyplot as plt
df_columns.boxplot("totalwords", by = "Author", grid = False, showfliers = False, color = "red")
plt.suptitle("")
plt.xlabel("")
plt.show()

The variation is huge. Some columns have 5k words. I need to keep that in mind, to see if it's something I want to deal with later on.

Importing the letters now

In [ ]:
df_letter = pd.read_excel("/content/finalized_mrr_letters.xlsx")

In [ ]:
df_letter.info()

In [ ]:
df_letter.head()

In [ ]:
df_letter['Text'] = df_letter['letter_content'].str.replace('\n', ' ', regex=False)

df_letter['Text'] = df_letter['letter_content'].str.replace(r'\s+', ' ', regex=True).str.strip()

In [ ]:
df_letter['totalwords'] = df_letter['Text'].str.split().str.len()

print(df_letter['totalwords'].sum())

In [ ]:
df_letter.info()

In [ ]:
df_columns.info()

Sine I am working with a model from HuggingFace 🤗 I need convert the pandas dataframe into an appropiate format.

In [ ]:
#combining the textual content to fine-tune
df_all = pd.concat([df_letter, df_columns], keys=['letter', 'column'], names=['Source', 'Row_ID']).reset_index()

df_final = df_all[['Text', 'Source']]

In [ ]:
df_final.head()

In [ ]:
df_final.info()

In [ ]:
from datasets import Dataset

In [ ]:
hf_dataset = Dataset.from_pandas(df_final)

In [ ]:
display(hf_dataset)

# Preprocessing - Tokenizing

The first step is tokenization. I will be borrowing the code from Tunstall et al. (2022) for this step. https://www.oreilly.com/library/view/natural-language-processing/9781098136789/

DistilBERT will be used for this part of my research project. This model achieves comparable performance to its larger cousin BERT, while being smaller. The advantage is that DistilBERT can be trained much quicker.

The opinion columns + letters will now be tokenized and encoded as numerical vectors using the tokenizer for DistilBERT.

In [ ]:
from transformers import DistilBertTokenizer

In [ ]:
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

In [ ]:
#I will now tokenize the whole dataset, first make a function

def tokenize(batch):
  return tokenizer(batch["Text"],
                   padding = "max_length",
                   truncation = True,
                   max_length = 512,
                   return_overflowing_tokens = True,
                   stride = 128) #need to work on this later to account for large columns. edit: has been worked on, good enough now!



By implementing return_overflowing_tokens the model can account for the token limit. It allows for the tokenizer to return tokens that exceed the max length, splitting the text into chunks.

I could experiment with other methods chunking methods... but this should work for now.

In [ ]:
#now map the function accross my dataset with .map

tokenized_dataset = hf_dataset.map(
    tokenize,
    batched = True,
    remove_columns = hf_dataset.column_names #this accounts for the overflowing tokens
)

In [ ]:
print(len(tokenized_dataset))

In [ ]:
total_tokens = sum(len(ids) for ids in tokenized_dataset["input_ids"])

print(f"Total Chunks: {len(tokenized_dataset)}")
print(f"Total Tokens: {total_tokens:,}")

# Fine-Tuning


For this step, I will use masked language modeling (MLM) to fine-tune DistilBERT on my dataset.

Although the article by Xiao and Feng (2025) that inspires my method, does not fine-tune their models I think that for my specific data it is the best way to go about it. Since the purpose of the authors there was just to develop an anomaly detection benchmark, my objectives are slightly different.

MLM is useful for fine-tuning on domain-specific text.


I will use these tutorials as my guide for this stage: https://huggingface.co/docs/transformers/tasks/masked_language_modeling

https://ai.plainenglish.io/fine-tuning-a-masked-language-model-for-domain-specific-understanding-a74b556cc49d


In [ ]:
#check the tokenizer being used

print(tokenizer)

In [ ]:
from transformers import AutoModelForMaskedLM

In [ ]:
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
model = AutoModelForMaskedLM.from_pretrained("distilbert-base-uncased")

In [ ]:
from transformers import DataCollatorForLanguageModeling

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm_probability=0.15 #15% of the tokens in each sequence will be placed with [MASK] and the model will guess or predict the likely token
)

In [ ]:
split_dataset = tokenized_dataset.train_test_split(test_size=0.1, seed=42)

train_ds = split_dataset["train"]
eval_ds = split_dataset["test"]

print(f"Training chunks: {len(train_ds)}")
print(f"Evaluation chunks: {len(eval_ds)}")

In [ ]:
samples = [tokenized_dataset[i] for i in range(2)]

chunked_check = data_collator(samples)

for chunk in chunked_check["input_ids"]:
    print(f"\n'>>> {tokenizer.decode(chunk)}'")

Output shows how [MASK] becomes a token that the model will try to predict.

Define the training hyperparamenters.

In [ ]:
from transformers import TrainingArguments

In [ ]:
batch_size = 16

training_args = TrainingArguments(
    output_dir = "distilbert-finetuned-mrr",
    num_train_epochs = 10,
    learning_rate = 2e-5,
    weight_decay = 0.01,
    per_device_train_batch_size = batch_size,

    eval_strategy = "epoch",
    logging_strategy = "steps",
    logging_steps = 50,

    load_best_model_at_end = True,
    save_strategy = "epoch"
)

Initiate the Trainer to wrap everything together

In [ ]:
from transformers import Trainer

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
)

In [ ]:
trainer.train()

In [ ]:
from huggingface_hub import login
login(token=#"redacted") #remember to redact this!!****

In [ ]:
model_name = "mrr-punk-columns-and-letters-distilbert"

In [ ]:
trainer.push_to_hub(f"limhayi/{model_name}")
tokenizer.push_to_hub(f"limhayi/{model_name}")

In [ ]:
trainer.model.push_to_hub("limhayi/mrr-punk-columns-and-letters-distilbert")

# Playing with the model and its predictions

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, AutoModelForMaskedLM

model_name = "limhayi/mrr-punk-columns-and-letters-distilbert"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMaskedLM.from_pretrained(model_name)

In [ ]:
import torch

test_text = "The genre of this album is [MASK]."

inputs = tokenizer(test_text, return_tensors="pt").to("cuda")
model.to("cuda")

with torch.no_grad():
    token_logits = model(**inputs).logits

mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]
mask_token_logits = token_logits[0, mask_token_index, :]
top_5_tokens = torch.topk(mask_token_logits, 5, dim=1).indices[0].tolist()

for token in top_5_tokens:
    print(f"'>>> {test_text.replace(tokenizer.mask_token, tokenizer.decode([token]))}'")

In [ ]:
test_text_2 = "This record is total [MASK], don't waste your money."


In [ ]:
inputs = tokenizer(test_text_2, return_tensors="pt").to("cuda") # add .to("cuda") if using GPU
model.to("cuda")

with torch.no_grad():
    token_logits = model(**inputs).logits

mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]
mask_token_logits = token_logits[0, mask_token_index, :]
top_5_tokens = torch.topk(mask_token_logits, 5, dim=1).indices[0].tolist()

for token in top_5_tokens:
    print(f"'>>> {test_text_2.replace(tokenizer.mask_token, tokenizer.decode([token]))}'")

In [ ]:
from transformers import pipeline

punk_brain = pipeline("fill-mask", model=model, tokenizer=tokenizer)

vanilla_brain = pipeline("fill-mask", model="distilbert-base-uncased")

text = "The band played a fast set at the [MASK]."

print("--- FINE-TUNED DISTILBERT PREDICTIONS ---")
for res in punk_brain(text):
    print(f"{res['token_str']}: {res['score']:.4f}")

print("\n--- VANILLA DISTILBERT PREDICTIONS ---")
for res in vanilla_brain(text):
    print(f"{res['token_str']}: {res['score']:.4f}")

In [ ]:

text = "The scene is being ruined by [MASK] who don't care about the music."

print("--- FINE-TUNED DISTILBERT PREDICTIONS ---")
for res in punk_brain(text):
    print(f"{res['token_str']}: {res['score']:.4f}")

print("\n--- VANILLA DISTILBERT PREDICTIONS ---")
for res in vanilla_brain(text):
    print(f"{res['token_str']}: {res['score']:.4f}")

In [ ]:
text = "people are [MASK]."

print("--- FINE-TUNED DISTILBERT PREDICTIONS ---")
for res in punk_brain(text):
    print(f"{res['token_str']}: {res['score']:.4f}")

print("\n--- VANILLA DISTILBERT PREDICTIONS ---")
for res in vanilla_brain(text):
    print(f"{res['token_str']}: {res['score']:.4f}")

In [ ]:
text = "cops are [MASK]."

print("--- FINE-TUNED DISTILBERT PREDICTIONS ---")
for res in punk_brain(text):
    print(f"{res['token_str']}: {res['score']:.4f}")

print("\n--- VANILLA DISTILBERT PREDICTIONS ---")
for res in vanilla_brain(text):
    print(f"{res['token_str']}: {res['score']:.4f}")

In [ ]:
text = "politics are not [MASK]."

print("--- FINE-TUNED DISTILBERT PREDICTIONS ---")
for res in punk_brain(text):
    print(f"{res['token_str']}: {res['score']:.4f}")

print("\n--- VANILLA DISTILBERT PREDICTIONS ---")
for res in vanilla_brain(text):
    print(f"{res['token_str']}: {res['score']:.4f}")

In [ ]:
text = "I am writing this column from [MASK]."

print("--- FINE-TUNED DISTILBERT PREDICTIONS ---")
for res in punk_brain(text):
    print(f"{res['token_str']}: {res['score']:.4f}")

print("\n--- VANILLA DISTILBERT PREDICTIONS ---")
for res in vanilla_brain(text):
    print(f"{res['token_str']}: {res['score']:.4f}")

In [ ]:
text = "America is not [MASK]."

print("--- FINE-TUNED DISTILBERT PREDICTIONS ---")
for res in punk_brain(text):
    print(f"{res['token_str']}: {res['score']:.4f}")

print("\n--- VANILLA DISTILBERT PREDICTIONS ---")
for res in vanilla_brain(text):
    print(f"{res['token_str']}: {res['score']:.4f}")

In [ ]:
text = "Punk is not [MASK]."

print("--- FINE-TUNED DISTILBERT PREDICTIONS ---")
for res in punk_brain(text):
    print(f"{res['token_str']}: {res['score']:.4f}")

print("\n--- VANILLA DISTILBERT PREDICTIONS ---")
for res in vanilla_brain(text):
    print(f"{res['token_str']}: {res['score']:.4f}")

In [ ]:
text = "vote for [MASK]."

print("--- FINE-TUNED DISTILBERT PREDICTIONS ---")
for res in punk_brain(text):
    print(f"{res['token_str']}: {res['score']:.4f}")

print("\n--- VANILLA DISTILBERT PREDICTIONS ---")
for res in vanilla_brain(text):
    print(f"{res['token_str']}: {res['score']:.4f}")

In [ ]:
text = "anarchy is [MASK]."

print("--- FINE-TUNED DISTILBERT PREDICTIONS ---")
for res in punk_brain(text):
    print(f"{res['token_str']}: {res['score']:.4f}")

print("\n--- VANILLA DISTILBERT PREDICTIONS ---")
for res in vanilla_brain(text):
    print(f"{res['token_str']}: {res['score']:.4f}")

In [ ]:
text = "the hardcore scene is [MASK]."

print("--- FINE-TUNED DISTILBERT PREDICTIONS ---")
for res in punk_brain(text):
    print(f"{res['token_str']}: {res['score']:.4f}")

print("\n--- VANILLA DISTILBERT PREDICTIONS ---")
for res in vanilla_brain(text):
    print(f"{res['token_str']}: {res['score']:.4f}")